# Dalux Build API - Complete Demo

## Setup: Configure API Client

In [1]:
!uv add pandas --active -q

In [2]:
# Add local package to path
import sys
from pathlib import Path

project_root = Path.cwd()
python_dir = project_root / "python"

if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))
    print(f"✓ Added {python_dir} to Python path")

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

BASE_URL = os.getenv("DALUX_BASE_URL")
API_KEY = os.getenv("DALUX_API_KEY")

if BASE_URL and API_KEY:
    print(f"✓ API credentials loaded from environment")
else:
    print(f"⚠️  Set DALUX_BASE_URL and DALUX_API_KEY environment variables")

✓ Added /Users/brunoadam/Documents/development/github/dalux-build/python/notebooks/python to Python path
✓ API credentials loaded from environment


In [3]:
# Create Dalux client with Pydantic support
from dalux_build import create_client

dalux = create_client()

In [4]:
import dalux_build
dalux_build.__version__

'2.1.4'

In [5]:
projects = dalux.projects.list_projects()

/var/folders/d1/x6l5g3q17xl7sxts9r454k6r0000gn/T/ipykernel_49983/3877746552.py:1: DeprecationWarning: list_projects() is deprecated and only returns the first page of results. Use get_projects() instead to fetch all projects with pagination.
  projects = dalux.projects.list_projects()


In [6]:
if len(projects) == 0:
    print("⚠️  No projects found for the current user")
    raise SystemExit("Please check your API credentials and try again.")
elif len(projects) == 1:
    print(f"✓ Found 1 project: {projects[0].project_name}")
    PROJECT_ID = projects[0].project_id
    dalux.set_default_project(PROJECT_ID)
else:
    PROJECT_ID = os.getenv("DALUX_PROJECT_ID")
    dalux.set_default_project(PROJECT_ID)

print(f"✓ Using project ID: {PROJECT_ID}")

✓ Found 1 project: LLYN.B250_GMP-Facility – Turnkey Contractor
✓ Using project ID: S313578016888324096


In [7]:
file_areas = dalux.file_areas.get_file_areas()
file_areas

[FileArea(file_area_id='S313578021116182528', file_area_name='Files', file_area_type='files'),
 FileArea(file_area_id='S313578021132959744', file_area_name='Shared files', file_area_type='shared'),
 FileArea(file_area_id='S313578021149736960', file_area_name='Published files', file_area_type='published')]

In [8]:
dalux.set_default_file_area(file_areas[0].file_area_id)

In [ ]:
# Path format: FileArea/Folder1/Folder2/...
# File area ID is resolved automatically from the file area name
all_files = dalux.files.get_files_in_folder(
    path="Files/4_Design/C07_Geometry/C07.11_Sketch",
    verbose=False
)

print(f"✓ Found {len(all_files)} files")
all_files

In [ ]:
version_sets = dalux.version_sets.get_version_sets(to_dataframe=True)
version_sets

In [ ]:
from dalux_build.models import FileNameFilter

filters = FileNameFilter(
    extensions=["ifc"],
    contains=["A valid string to filter files by name here"],
)

downloaded_files = dalux.version_sets.download_files(
    identifiers=[filters],
    version_set_ids=["a valid version_set_id here"], # You need to provide a valid version_set_id
    save_path="downloads",
    verbose=True
)
downloaded_files

In [ ]:
# Get the list of versions sets and their files
version_sets = dalux.version_sets.get_version_sets(to_dataframe=True)

version_set_ids = version_sets["versionSetId"].tolist()
version_set_ids

In [ ]:
downloaded_files = dalux.version_sets.download_files(
    identifiers=[filters],
    version_set_ids=version_set_ids,
    save_path="downloads",
    verbose=True
)
downloaded_files

In [ ]:
version_set_files = dalux.version_sets.list_version_set_files(
    version_set_id="a valid version_set_id here"
)
version_set_files

In [ ]:
project_tasks = dalux.tasks.get_all_project_tasks()
project_tasks

In [ ]:
# Both formats work:
# Format 1 (explicit file area name): Files/Files/4_Design/...
# Format 2 (uses default file area): Files/4_Design/...

all_files_in_folder = dalux.files.get_files_in_folder(
    path="Files/4_Design/C07_Geometry/C07.05_BIM",  # Using Format 2
    verbose=True
)
print(f"\n✓ Found {len(all_files_in_folder)} files")
all_files_in_folder

In [11]:
# First, let's see what top-level folders exist
all_folders = dalux.folders.get_folders(file_area_id=file_areas[0].file_area_id, verbose=True)

# Find root-level folders (parent_folder_id is None or the file_area_id)
file_area_id = file_areas[0].file_area_id
root_folders = []
for folder in all_folders:
    folder_data = folder if isinstance(folder, dict) else folder.model_dump(by_alias=True)
    data = folder_data.get("data", folder_data)
    parent_id = data.get("parentFolderId") or data.get("parentId")
    # Root folders have parent_id == file_area_id or None
    if parent_id == file_area_id or parent_id is None:
        name = data.get("folderName") or data.get("name")
        folder_id = data.get("folderId") or data.get("id")
        if name:
            root_folders.append((name, folder_id))
            print(f"  {name}")

print(f"\nFound {len(root_folders)} root-level folders")

Fetching pages: 100%|██████████| 1936/1936 [00:01<00:00, 1030.02item/s]

  Files

Found 1 root-level folders


In [12]:
# Debug: Check folder IDs and hierarchy
def get_folder_info(folder):
    data = folder if isinstance(folder, dict) else folder.model_dump(by_alias=True)
    data = data.get("data", data)
    folder_id = data.get("folderId") or data.get("id")
    name = data.get("folderName") or data.get("name")
    parent_id = data.get("parentFolderId") or data.get("parentId")
    return folder_id, name, parent_id

# Find the 'Files' folder and check 4_Design's parent
files_folder_id = None
design_4_parent_id = None

for folder in all_folders:
    fid, name, parent = get_folder_info(folder)
    if name == 'Files':
        files_folder_id = fid
        print(f"'Files' folder ID: {fid}")
        print(f"'Files' parent ID: {parent}")
    if name == '4_Design':
        design_4_parent_id = parent
        print(f"'4_Design' folder ID: {fid}")
        print(f"'4_Design' parent ID: {parent}")

if files_folder_id == design_4_parent_id:
    print(f"\n✓ '4_Design' IS a child of 'Files'")
    print(f"  Use path: Files/Files/4_Design/...")
else:
    print(f"\n✗ '4_Design' is NOT a direct child of 'Files'")
    print(f"  '4_Design' parent: {design_4_parent_id}")
    # Find what folder has this ID
    for folder in all_folders:
        fid, name, parent = get_folder_info(folder)
        if fid == design_4_parent_id:
            print(f"  Parent folder name: {name}")

'Files' folder ID: S313578021070045184
'Files' parent ID: None
'4_Design' folder ID: S315408213870641152
'4_Design' parent ID: S313578021070045184

✓ '4_Design' IS a child of 'Files'
  Use path: Files/Files/4_Design/...


In [13]:
files = [
    "list of file paths here"
    ]

# Alternative way to download files using a list of file paths

if len(files) == 0:
    print("⚠️  No files found to download. Please check the file paths and try again.")
else:
    downloaded_files = dalux.files.bulk_download_files(
        files=files,
        save_metadata=True,
        verbose=True, 
        save_path="outputs/",
        save_historically=True,
    )

GET /5.0/projects/S313578016888324096/file_areas/S313578021116182528/files/list of file paths here


ApiError: API request failed: Parameter validation failed for fileId = list of file paths here

In [ ]:
FILE_AREA_ID = file_areas[0].file_area_id  # You need to provide a valid file_area_id
dalux.set_default_file_area(FILE_AREA_ID)
FILE_AREA_ID

In [ ]:
from dalux_build.models import FileNameFilter

filters = FileNameFilter(
    contains=["K00_T99"],  # Molio model's name convention
    extensions=[".ifc"]
)

# Uses default file area set earlier
dalux.files.bulk_download_folder(
    "Files/4_Design/C07_Geometry/C07.05_BIM",
    save_path="outputs",
    filters=filters,
    verbose=True
)

In [ ]:
from dalux_build.models import FileNameFilter

filters = FileNameFilter(
    contains=["K00_T99"], # Molio model's name convention
    extensions=[".ifc"]
)

dalux.files.bulk_download_folder(
    "Files/4_Design/C07_Geometry/C07.05_BIM",
    save_path="outputs",
    filters=filters,
    verbose=True
)